# Energy Modeling Concepts with topologic_fast

This notebook demonstrates energy modeling concepts using topologic_fast. It covers:

1. Creating building geometry (cells representing spaces)
2. Creating window apertures on external walls
3. Creating shading surfaces
4. Decomposing cell complexes to identify external/internal faces
5. Visualizing buildings with energy-related annotations

**Note:** Full energy simulation (OpenStudio integration) is not yet implemented in topologic_fast.
This notebook focuses on the geometric and topological aspects of energy modeling.

## Import Required Libraries

In [ ]:
# Import topologic_fast
import topologic_fast as tf

# Import visualization libraries
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
import math

print("topologic_fast imported successfully")
print(f"Available classes: Vertex, Edge, Wire, Face, Shell, Cell, CellComplex, etc.")

## Create Building Geometry

We'll create a simple multi-story building as a CellComplex where each Cell represents a thermal zone (space).

In [ ]:
# Building parameters
building_width = 20.0  # meters
building_length = 30.0  # meters
floor_height = 3.0  # meters
num_floors = 3

# Create cells for each floor
cells = []
for floor in range(num_floors):
    z_offset = floor * floor_height
    cell = tf.Cell.Box(
        origin_x=-building_width/2,
        origin_y=-building_length/2,
        origin_z=z_offset,
        width=building_width,
        length=building_length,
        height=floor_height
    )
    cells.append(cell)
    print(f"Floor {floor}: Volume = {cell.Volume():.2f} m^3")

# Create a cell complex from the cells
building = tf.CellComplex.ByCells(cells)
print(f"\nBuilding created with {building.NumCells()} cells")
print(f"Total volume: {building.Volume():.2f} m^3")

## Decompose Building Geometry

For energy modeling, we need to identify:
- External vertical faces (walls) - for glazing
- External horizontal faces (roof/floor slab) - for insulation
- Internal faces - for thermal zoning

In [ ]:
def decompose_building(building):
    """
    Decompose a building (CellComplex) into categorized faces.
    
    Returns a dictionary with:
    - external_vertical_faces: walls facing outside
    - external_horizontal_faces: roof and ground floor
    - internal_faces: faces shared between cells
    """
    all_faces = building.Faces()
    all_cells = building.Cells()
    
    external_vertical = []
    external_horizontal = []
    internal_faces = []
    
    for face in all_faces:
        # Get face normal to determine orientation
        normal = face.Normal()
        if normal is None:
            continue
            
        nx, ny, nz = normal
        
        # Count how many cells this face belongs to
        # NOTE: In topologic_fast, direct face-cell adjacency queries
        # are not yet fully implemented. We use a geometric approach.
        centroid = face.CenterOfMass()
        
        # Classify by normal direction
        is_horizontal = abs(nz) > 0.9  # Face is roughly horizontal
        is_vertical = abs(nz) < 0.1    # Face is roughly vertical
        
        # Check if external (on building boundary)
        cx, cy, cz = centroid
        tolerance = 0.01
        
        is_external = (
            abs(cx - (-building_width/2)) < tolerance or
            abs(cx - (building_width/2)) < tolerance or
            abs(cy - (-building_length/2)) < tolerance or
            abs(cy - (building_length/2)) < tolerance or
            abs(cz - 0) < tolerance or
            abs(cz - (num_floors * floor_height)) < tolerance
        )
        
        if is_external:
            if is_horizontal:
                external_horizontal.append(face)
            elif is_vertical:
                external_vertical.append(face)
        else:
            internal_faces.append(face)
    
    return {
        'external_vertical_faces': external_vertical,
        'external_horizontal_faces': external_horizontal,
        'internal_faces': internal_faces
    }

decomposition = decompose_building(building)
print(f"External vertical faces (walls): {len(decomposition['external_vertical_faces'])}")
print(f"External horizontal faces (roof/floor): {len(decomposition['external_horizontal_faces'])}")
print(f"Internal faces: {len(decomposition['internal_faces'])}")

## Create Window Apertures

For energy modeling, we create windows (apertures) on external walls using a glazing ratio.

In [ ]:
def create_apertures(walls, glazing_ratio=0.25):
    """
    Create window apertures by scaling external walls.
    
    Parameters:
    - walls: list of Face objects (external vertical faces)
    - glazing_ratio: window-to-wall ratio (0.0 to 1.0)
    
    Returns:
    - list of Face objects representing windows
    """
    apertures = []
    scale_factor = math.sqrt(glazing_ratio)
    
    for wall in walls:
        centroid = wall.CenterOfMass()
        cx, cy, cz = centroid
        
        # Create a scaled copy of the wall face as the aperture
        # NOTE: Direct face scaling is not yet implemented in topologic_fast.
        # We'll create apertures from the wall vertices.
        
        vertices = wall.Vertices()
        if len(vertices) >= 3:
            # Scale vertices towards centroid
            scaled_vertices = []
            for v in vertices:
                vx, vy, vz = v.Coordinates()
                new_x = cx + (vx - cx) * scale_factor
                new_y = cy + (vy - cy) * scale_factor
                new_z = cz + (vz - cz) * scale_factor
                scaled_vertices.append(tf.Vertex.ByCoordinates(new_x, new_y, new_z))
            
            # Create wire and face from scaled vertices
            wire = tf.Wire.ByVertices(scaled_vertices, close=True)
            aperture = tf.Face.ByWire(wire)
            apertures.append(aperture)
    
    return apertures

# Create apertures with 25% glazing ratio
glazing_ratio = 0.25
walls = decomposition['external_vertical_faces']
apertures = create_apertures(walls, glazing_ratio)

print(f"Created {len(apertures)} window apertures")
total_wall_area = sum(w.Area() for w in walls)
total_aperture_area = sum(a.Area() for a in apertures)
print(f"Total wall area: {total_wall_area:.2f} m^2")
print(f"Total aperture area: {total_aperture_area:.2f} m^2")
print(f"Actual glazing ratio: {total_aperture_area/total_wall_area:.2%}")

## Create Shading Surfaces

Shading surfaces (overhangs, fins) affect solar heat gain. We'll create simple horizontal overhangs above windows.

In [ ]:
def create_overhangs(apertures, overhang_depth=1.0):
    """
    Create horizontal overhangs above window apertures.
    
    Parameters:
    - apertures: list of Face objects (windows)
    - overhang_depth: depth of overhang in meters
    
    Returns:
    - list of Face objects representing overhangs
    """
    overhangs = []
    
    for aperture in apertures:
        # Get aperture bounding box
        bbox = aperture.BoundingBox()
        (min_x, min_y, min_z), (max_x, max_y, max_z) = bbox
        
        # Get aperture normal to determine overhang direction
        normal = aperture.Normal()
        if normal is None:
            continue
        nx, ny, nz = normal
        
        # Create overhang above the aperture
        # The overhang extends outward from the wall
        overhang_z = max_z + 0.1  # Slightly above window top
        
        # Create a rectangular overhang
        width = max_x - min_x if abs(ny) > 0.5 else max_y - min_y
        
        # Create overhang vertices
        v1 = tf.Vertex.ByCoordinates(min_x, min_y, overhang_z)
        v2 = tf.Vertex.ByCoordinates(max_x, min_y, overhang_z)
        v3 = tf.Vertex.ByCoordinates(max_x + nx * overhang_depth, 
                                      min_y + ny * overhang_depth, 
                                      overhang_z)
        v4 = tf.Vertex.ByCoordinates(min_x + nx * overhang_depth, 
                                      min_y + ny * overhang_depth, 
                                      overhang_z)
        
        wire = tf.Wire.ByVertices([v1, v2, v3, v4], close=True)
        overhang = tf.Face.ByWire(wire)
        overhangs.append(overhang)
    
    return overhangs

# Create overhangs
shading_surfaces = create_overhangs(apertures, overhang_depth=0.5)
print(f"Created {len(shading_surfaces)} shading surfaces (overhangs)")
total_shading_area = sum(s.Area() for s in shading_surfaces)
print(f"Total shading area: {total_shading_area:.2f} m^2")

## Assign Simulated Energy Values

In a real energy simulation, we would run OpenStudio/EnergyPlus. Here we simulate some results.

In [ ]:
# NOTE: EnergyModel.ByTopology and simulation features are not yet implemented
# in topologic_fast. This section demonstrates how results would be used.

# Simulate energy values for each cell (thermal zone)
import random
random.seed(42)  # For reproducibility

cells = building.Cells()
energy_data = []

for i, cell in enumerate(cells):
    # Simulated values (in reality, these come from EnergyPlus)
    cooling_load = random.uniform(500, 1500)  # Watts
    heating_load = random.uniform(300, 1000)  # Watts
    
    energy_data.append({
        'cell_index': i,
        'volume': cell.Volume(),
        'area': cell.Area(),
        'cooling_load': cooling_load,
        'heating_load': heating_load,
        'center': cell.CenterOfMass()
    })
    
    print(f"Zone {i}: Cooling={cooling_load:.1f}W, Heating={heating_load:.1f}W")

# Calculate totals
total_cooling = sum(d['cooling_load'] for d in energy_data)
total_heating = sum(d['heating_load'] for d in energy_data)
print(f"\nTotal Building: Cooling={total_cooling:.1f}W, Heating={total_heating:.1f}W")

## Visualize Building with Energy Data

We'll create an interactive 3D visualization showing the building, apertures, and color-coded energy values.

In [ ]:
def create_mesh_data(face):
    """Extract mesh data from a face for Plotly visualization."""
    vertices = face.Vertices()
    if len(vertices) < 3:
        return None, None, None
    
    # Get vertex coordinates
    coords = [v.Coordinates() for v in vertices]
    
    # Simple fan triangulation for convex polygons
    x = [c[0] for c in coords]
    y = [c[1] for c in coords]
    z = [c[2] for c in coords]
    
    # Create triangle indices (fan from first vertex)
    i_idx = []
    j_idx = []
    k_idx = []
    
    for idx in range(1, len(coords) - 1):
        i_idx.append(0)
        j_idx.append(idx)
        k_idx.append(idx + 1)
    
    return (x, y, z), (i_idx, j_idx, k_idx)

def visualize_building(building, apertures, shading_surfaces, energy_data):
    """Create interactive 3D visualization of building with energy data."""
    fig = go.Figure()
    
    # Get all faces from the building
    faces = building.Faces()
    
    # Combine all mesh data for building faces
    all_x, all_y, all_z = [], [], []
    all_i, all_j, all_k = [], [], []
    
    for face in faces:
        coords, indices = create_mesh_data(face)
        if coords is None:
            continue
        
        offset = len(all_x)
        all_x.extend(coords[0])
        all_y.extend(coords[1])
        all_z.extend(coords[2])
        all_i.extend([i + offset for i in indices[0]])
        all_j.extend([j + offset for j in indices[1]])
        all_k.extend([k + offset for k in indices[2]])
    
    # Add building mesh
    fig.add_trace(go.Mesh3d(
        x=all_x, y=all_y, z=all_z,
        i=all_i, j=all_j, k=all_k,
        color='lightgray',
        opacity=0.3,
        name='Building Envelope',
        flatshading=True
    ))
    
    # Add apertures (windows)
    apt_x, apt_y, apt_z = [], [], []
    apt_i, apt_j, apt_k = [], [], []
    
    for aperture in apertures:
        coords, indices = create_mesh_data(aperture)
        if coords is None:
            continue
        
        offset = len(apt_x)
        apt_x.extend(coords[0])
        apt_y.extend(coords[1])
        apt_z.extend(coords[2])
        apt_i.extend([i + offset for i in indices[0]])
        apt_j.extend([j + offset for j in indices[1]])
        apt_k.extend([k + offset for k in indices[2]])
    
    if apt_x:
        fig.add_trace(go.Mesh3d(
            x=apt_x, y=apt_y, z=apt_z,
            i=apt_i, j=apt_j, k=apt_k,
            color='lightblue',
            opacity=0.7,
            name='Windows',
            flatshading=True
        ))
    
    # Add shading surfaces
    shd_x, shd_y, shd_z = [], [], []
    shd_i, shd_j, shd_k = [], [], []
    
    for shade in shading_surfaces:
        coords, indices = create_mesh_data(shade)
        if coords is None:
            continue
        
        offset = len(shd_x)
        shd_x.extend(coords[0])
        shd_y.extend(coords[1])
        shd_z.extend(coords[2])
        shd_i.extend([i + offset for i in indices[0]])
        shd_j.extend([j + offset for j in indices[1]])
        shd_k.extend([k + offset for k in indices[2]])
    
    if shd_x:
        fig.add_trace(go.Mesh3d(
            x=shd_x, y=shd_y, z=shd_z,
            i=shd_i, j=shd_j, k=shd_k,
            color='green',
            opacity=0.8,
            name='Shading',
            flatshading=True
        ))
    
    # Add energy data markers at cell centroids
    cooling_values = [d['cooling_load'] for d in energy_data]
    centers = [d['center'] for d in energy_data]
    
    fig.add_trace(go.Scatter3d(
        x=[c[0] for c in centers],
        y=[c[1] for c in centers],
        z=[c[2] for c in centers],
        mode='markers+text',
        marker=dict(
            size=15,
            color=cooling_values,
            colorscale='Thermal',
            colorbar=dict(title='Cooling Load (W)'),
            showscale=True
        ),
        text=[f"Zone {d['cell_index']}<br>Cooling: {d['cooling_load']:.0f}W" for d in energy_data],
        name='Thermal Zones'
    ))
    
    # Update layout
    fig.update_layout(
        title='Building Energy Model Visualization',
        scene=dict(
            aspectmode='data',
            xaxis_title='X (m)',
            yaxis_title='Y (m)',
            zaxis_title='Z (m)'
        ),
        width=900,
        height=700,
        showlegend=True
    )
    
    return fig

# Create and display visualization
fig = visualize_building(building, apertures, shading_surfaces, energy_data)
fig.show()

## Floor-by-Floor Analysis

Create a visualization showing energy values mapped to floor plans.

In [ ]:
def get_floor_faces(building, floor_index, floor_height=3.0):
    """
    Get the bottom faces of cells on a specific floor.
    These represent the floor plan.
    """
    cells = building.Cells()
    floor_faces = []
    
    target_z = floor_index * floor_height + 0.01  # Slightly above floor level
    
    for cell in cells:
        centroid = cell.CenterOfMass()
        cell_z = centroid[2]
        
        # Check if this cell is on the target floor
        if floor_index * floor_height <= cell_z < (floor_index + 1) * floor_height:
            # Get bottom face (horizontal face with lowest z)
            faces = cell.Faces()
            for face in faces:
                normal = face.Normal()
                if normal and abs(normal[2]) > 0.9:  # Horizontal face
                    face_z = face.CenterOfMass()[2]
                    if face_z < cell_z:  # Bottom face
                        floor_faces.append(face)
                        break
    
    return floor_faces

# Create subplots for each floor
fig = make_subplots(
    rows=1, cols=num_floors,
    subplot_titles=[f'Floor {i}' for i in range(num_floors)],
    specs=[[{'type': 'xy'} for _ in range(num_floors)]]
)

for floor in range(num_floors):
    floor_faces = get_floor_faces(building, floor, floor_height)
    cooling = energy_data[floor]['cooling_load']
    
    # Simple rectangle for floor plan
    fig.add_trace(
        go.Scatter(
            x=[-building_width/2, building_width/2, building_width/2, -building_width/2, -building_width/2],
            y=[-building_length/2, -building_length/2, building_length/2, building_length/2, -building_length/2],
            mode='lines',
            fill='toself',
            fillcolor=f'rgba({int(cooling/1500*255)}, {int((1-cooling/1500)*255)}, 100, 0.5)',
            line=dict(color='black', width=2),
            name=f'Floor {floor}',
            showlegend=True
        ),
        row=1, col=floor+1
    )
    
    # Add annotation
    fig.add_annotation(
        x=0, y=0,
        text=f"Cooling: {cooling:.0f}W",
        showarrow=False,
        row=1, col=floor+1
    )

fig.update_layout(
    title='Floor-by-Floor Cooling Load Analysis',
    height=400,
    width=900
)

fig.show()

## Summary

This notebook demonstrated energy modeling concepts using topologic_fast:

1. **Building Geometry**: Created multi-story buildings as CellComplexes
2. **Face Decomposition**: Identified external walls, roof/floor, and internal partitions
3. **Apertures**: Created window openings using a glazing ratio
4. **Shading**: Generated overhang shading surfaces
5. **Visualization**: Created interactive 3D views with energy data

### Features Not Yet Implemented in topologic_fast

- `EnergyModel.ByTopology()` - Create OpenStudio energy model
- `EnergyModel.Run()` - Run EnergyPlus simulation
- `EnergyModel.Query()` - Query simulation results from SQL
- `Honeybee.ModelByTopology()` - Create Honeybee model
- `Topology.AddApertures()` - Add apertures to faces
- `CellComplex.Decompose()` - Automatic face classification

These features are planned for future releases.

In [ ]:
# Clean up
tf.clear_store()
print("Topology store cleared.")